# Baseline Waste Classifier

This notebook is the first experiment entry point for the waste computer vision project.

Goal: train a simple image classifier for residential waste categories and record baseline metrics.

Keep patent-sensitive implementation details out of this public notebook until the filing strategy is clear.

## 1. Runtime Check

In Colab, set `Runtime > Change runtime type > GPU` when training models.

In [ ]:
import sys
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Clone or Update Repository

Run this cell only if you opened a blank Colab notebook. If you opened this notebook directly from GitHub, you may skip it.

In [ ]:
# In Colab only:
# !git clone https://github.com/Ishank-dubey/waste-cv-classifier.git
# %cd waste-cv-classifier

# If already cloned:
# !git pull

## 3. Install/Import Dependencies

In [ ]:
# Colab usually has most of these installed already.
# !pip install -q -r requirements.txt

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 4. Dataset Layout

Use an image-folder structure like this:

```text
data/
├── train/
│   ├── organic/
│   ├── plastic/
│   ├── paper_cardboard/
│   ├── metal/
│   ├── glass/
│   └── other/
├── val/
│   ├── organic/
│   ├── plastic/
│   ├── paper_cardboard/
│   ├── metal/
│   ├── glass/
│   └── other/
└── test/
    ├── organic/
    ├── plastic/
    ├── paper_cardboard/
    ├── metal/
    ├── glass/
    └── other/
```

Datasets should not be committed to GitHub.

In [ ]:
DATA_DIR = Path('data')
TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR = DATA_DIR / 'val'
TEST_DIR = DATA_DIR / 'test'

EXPECTED_CLASSES = ['organic', 'plastic', 'paper_cardboard', 'metal', 'glass', 'other']

print('Train directory exists:', TRAIN_DIR.exists())
print('Validation directory exists:', VAL_DIR.exists())
print('Test directory exists:', TEST_DIR.exists())
print('Expected classes:', EXPECTED_CLASSES)

## 5. Data Loaders

In [ ]:
image_size = 224
batch_size = 32

train_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

if TRAIN_DIR.exists() and VAL_DIR.exists():
    train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
    val_ds = datasets.ImageFolder(VAL_DIR, transform=val_tfms)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    class_names = train_ds.classes
    print('Classes:', class_names)
    print('Class to index mapping:', train_ds.class_to_idx)
    if sorted(class_names) != sorted(EXPECTED_CLASSES):
        print('Warning: class folders do not match EXPECTED_CLASSES.')
    print('Train images:', len(train_ds))
    print('Validation images:', len(val_ds))
else:
    print('Dataset not found yet. Add images using the folder layout above before training.')

## 6. Model

Start with transfer learning using ResNet-18. This is not the final research model; it is the baseline.

In [ ]:
def build_model(num_classes):
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

if 'class_names' in globals():
    model = build_model(len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    print(model.__class__.__name__, 'ready')

## 7. Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, np.array(all_labels), np.array(all_preds)

In [ ]:
epochs = 5

if 'model' in globals():
    history = []
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, y_true, y_pred = evaluate(model, val_loader, criterion)
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
        })
        print(f'Epoch {epoch}: train_acc={train_acc:.3f}, val_acc={val_acc:.3f}')

    pd.DataFrame(history)

## 8. Evaluation Report

In [ ]:
if 'y_true' in globals():
    print(classification_report(y_true, y_pred, target_names=class_names))
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(ax=ax, xticks_rotation=45, cmap='Blues')
    plt.title('Baseline Waste Classifier - Confusion Matrix')
    plt.show()

## 9. Save Baseline Model

Model files should be treated as local experiment outputs and not committed to GitHub.

In [ ]:
if 'model' in globals():
    output_dir = Path('outputs')
    output_dir.mkdir(exist_ok=True)
    torch.save({
        'model_state_dict': model.state_dict(),
        'class_names': class_names,
        'history': history,
    }, output_dir / 'baseline_resnet18.pt')
    print('Saved to outputs/baseline_resnet18.pt')